In [1]:
import sys
sys.path.insert(0, '/media/raid0/US_FiLMUNet')

import torch
from safetensors.torch import load_file
from nets.unet_attn import UNet2DAttn
from nets.segm_net import UNet2DFiLM


In [ ]:
model = UNet2DAttn(
    in_channels=3,
    num_classes=1,
    n_organs=10,
    size=32,
    depth=5,
    attn_start=0,
    use_attn=True,
    img_size=512,
    patch_size=8,
    emb_dim=768,
    n_heads=8,
    distill=False,
    use_dwt=False,
    wavelet='haar',
    use_shape=False,
    shape_res=64,
)

In [ ]:
model = UNet2DFiLM(
    in_channels=3,
    num_classes=1,
    # n_organs=len(organ_to_class_dict),
    n_organs=8,
    size=32,
    depth=5,
    film_start=0,
    use_film=True,
    distill=False
)

In [ ]:
CHECKPOINT = '/media/raid0/US_FiLMUNet/checkpoints/unet5/model.safetensors'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

state_dict = load_file(CHECKPOINT)
state_dict = {k: v for k, v in state_dict.items() if 'distill' not in k}
load_result = model.load_state_dict(state_dict, strict=False)
print(load_result)

model.eval()
model.to(DEVICE)
print(f"Model loaded on {DEVICE}")

In [ ]:
model.eval()
model.to(DEVICE)
print(f"Model loaded on {DEVICE}")

In [78]:
from torch.utils.data import DataLoader
from data_classes.datasets import USdatasetOmni
from utils.paths import DATA_DIR
from utils.utils import get_sft_transforms

BATCH_SIZE = 8

train_dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split='train',
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type='segmentation',
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=False,
    id_dropout=0.0,
)

test_dataset = USdatasetOmni(
    base_dir=DATA_DIR,
    split='val_cls',
    transforms=get_sft_transforms(train=False),
    out_size=512,
    data_type='segmentation',
    ccl_crop=False,
    keep_aspect_ratio=True,
    self_norm=False,
    id_dropout=0.0,
    skip_dataset='Testicle'
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Test: {len(test_dataset)} images")

Train: 5219 images | Test: 1931 images


In [ ]:
import pprint


cnt = {}
for item in test_dataset.items:
    if item['organ_label'] in cnt.keys():
        cnt[item['organ_label']] += 1
    else:
        cnt[item['organ_label']] = 1

pprint.pp(cnt)

{'breast_luminal': 386,
 'kidney': 67,
 'thyroid': 428,
 'testicle': 853,
 'fetal': 121,
 'cardiac': 76}


In [80]:
from data_classes.datasets import defaultdict
import random
import copy

label_to_indices = defaultdict(list)
for idx, item in enumerate(test_dataset.items):
    if item['organ_label'] == 'testicle': continue
    if item['organ_label'] == 'breast': item['organ_label'] = 'breast_luminal'
    label_to_indices[item['organ_label']].append(idx)

unique_labels = list(label_to_indices.keys())
num_classes = len(unique_labels)
N = int(len([i for i in test_dataset.items if i['organ_label'] != 'testicle']) * 0.1)
samples_per_label = N // num_classes


sampled_indices = []

for label in unique_labels:
    sampled_indices.extend(random.sample(label_to_indices[label], k=samples_per_label))

val_dataset = copy.deepcopy(test_dataset)
val_dataset.items = []

for idx in sampled_indices:
    val_dataset.items.append(test_dataset[idx])

for i, item in enumerate(test_dataset.items[:]):
    if i in sampled_indices:
        test_dataset.items.remove(item)



In [81]:
len(val_dataset.items)

105

In [82]:
len(test_dataset.items)

1826

['breast_luminal', 'kidney', 'thyroid', 'fetal', 'cardiac']

In [64]:
import random
sampled_indices = []

for label in unique_labels:
    sampled_indices.extend(random.sample(label_to_indices[label], k=samples_per_label))

In [ ]:
from tqdm import tqdm

gammas, betas = [], []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        pixel_values = batch['pixel_values'].to(DEVICE)
        organ_id = batch['organ_id'].long().to(DEVICE)
        forward_ctx = model._prepare_forward(
            pixel_values=pixel_values,
            organ_id=organ_id,
        )
        # mod_list[l] = (gamma (B, C, 1, 1), beta (B, C, 1, 1))
        mod_list = forward_ctx['mod_list']
        n_layers = len(mod_list)
        B = pixel_values.shape[0]
        for img_idx in range(B):
            img_gammas = [mod_list[l][0][img_idx, :, 0, 0].cpu() for l in range(n_layers)]
            img_betas  = [mod_list[l][1][img_idx, :, 0, 0].cpu() for l in range(n_layers)]
            gammas.append(img_gammas)
            betas.append(img_betas)
# return gammas, betas

In [ ]:
from tqdm import tqdm


def collect_modulations(model, loader, device):
    """
    Run inference and collect FiLM gamma and beta for each layer, per image.

    Returns:
        gammas: list[n_images] of list[n_layers] of (C,) cpu tensors
        betas:  list[n_images] of list[n_layers] of (C,) cpu tensors
    """
    gammas, betas = [], []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader):
            pixel_values = batch['pixel_values'].to(device)
            organ_id = batch['organ_id'].long().to(device)

            forward_ctx = model._prepare_forward(
                pixel_values=pixel_values,
                organ_id=organ_id,
            )
            # mod_list[l] = (gamma (B, C, 1, 1), beta (B, C, 1, 1))
            mod_list = forward_ctx['mod_list']
            n_layers = len(mod_list)
            B = pixel_values.shape[0]

            for img_idx in range(B):
                img_gammas = [mod_list[l][0][img_idx, :, 0, 0].cpu() for l in range(n_layers)]
                img_betas  = [mod_list[l][1][img_idx, :, 0, 0].cpu() for l in range(n_layers)]
                gammas.append(img_gammas)
                betas.append(img_betas)

    return gammas, betas


gammas_train, betas_train = collect_modulations(model, train_loader, DEVICE)
gammas_test,  betas_test  = collect_modulations(model, test_loader,  DEVICE)

print(f"Train: {len(gammas_train)} images, {len(gammas_train[0])} layers each")
print(f"Test:  {len(gammas_test)} images,  {len(gammas_test[0])} layers each")
print(f"Layer 0 gamma shape: {gammas_train[0][0].shape}")
print(f"Layer 0 beta  shape: {betas_train[0][0].shape}")